In [37]:
%pip install opendatasets ultralytics albumentations torchmetrics --quiet

In [38]:
import opendatasets as od
od.download_kaggle_dataset("https://www.kaggle.com/competitions/find-the-seagulls/data", "")

Skipping, found downloaded files in "find-the-seagulls" (use force=True to force download)


In [39]:
import os
from tqdm import tqdm
import yaml

In [40]:
import numpy as np
import pandas as pd

In [41]:
import matplotlib.pyplot as plt
from PIL import Image

In [42]:
import cv2

import albumentations as A
from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

cuda


# Подготовка данных

In [43]:
DATA_DIR = '/content/find-the-seagulls/data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')
LABELS_DIR = os.path.join(TRAIN_DIR, 'labels')
IMAGES_DIR = os.path.join(TRAIN_DIR, 'images')

In [44]:
IMAGE_SIZE = 640

In [45]:
yaml_content = {
    'path': str(DATA_DIR),
    'train': 'train/images',
    'val': 'train/images',
    'nc': 1,
    'names': {0: 'seagull'}
}

yaml_path = DATA_DIR + "/seagulls.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

# Детекция с помощью YOLO

In [ ]:
model = YOLO('yolov8m.pt')

results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=IMAGE_SIZE,
    batch=32,
    device=DEVICE,
    optimizer='AdamW',
    lr0=1e-3,
    lrf=1e-5,
    weight_decay=1e-4,
    augment=True,
    patience=10,
    save=True,
    exist_ok=True,
    verbose=True
)

best_model = YOLO('/content/runs/detect/train/weights/best.pt')

engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/find-the-seagulls/data/seagulls.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=1e-05, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=10, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, projec

In [ ]:
def xyxy_to_yolo_norm(xyxy, img_w, img_h):
    x1, y1, x2, y2 = xyxy

    x_c = (x1 + x2) / 2
    y_c = (y1 + y2) / 2
    w = x2 - x1
    h = y2 - y1

    x_c_norm = x_c / img_w
    y_c_norm = y_c / img_h
    w_norm = w / img_w
    h_norm = h / img_h

    return f"0 {x_c_norm:.4f} {y_c_norm:.4f} {w_norm:.4f} {h_norm:.4f}"

def format_bbox_string(boxes, img_w, img_h):
    if boxes is None or len(boxes) == 0:
        return "-1"

    bbox_parts = []
    for box in boxes:
        xyxy = box.xyxy[0].cpu().numpy()
        yolo_str = xyxy_to_yolo_norm(xyxy, img_w, img_h)
        bbox_parts.append(yolo_str)

    return " ".join(bbox_parts)

In [ ]:
best_model.eval()

test_files = sorted(
    [f for f in os.listdir(TEST_DIR + "/images") if f.lower().endswith(('.jpg', '.png', '.jpeg'))],
    key=lambda x: int(''.join(filter(str.isdigit, x.split('.')[0])) or 0)
)

predictions = []

for idx, fname in enumerate(tqdm(test_files, desc='Creating submission')):
    img_path = str(TEST_DIR + "/images/" + fname)

    results = best_model.predict(
        source=img_path,
        imgsz=IMAGE_SIZE,
        verbose=False,
        save=False
    )

    boxes = results[0].boxes if results else None
    bbox_str = format_bbox_string(boxes, IMAGE_SIZE, IMAGE_SIZE)

    predictions.append({
        'index': idx,
        'filename': fname,
        'bbox': bbox_str
    })

submission = pd.DataFrame(predictions)
submission.to_csv('submission.csv', index=False)